In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchaudio


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [19]:
DATA_DIR  = "DATA"
VIDEO_DIR = os.path.join(DATA_DIR, "raw_videos")
IMAGE_DIR = os.path.join(DATA_DIR, "images")
AUDIO_DIR = os.path.join(DATA_DIR, "audio")


In [20]:
emotion_map = {
    "01": 0,  # neutral
    "02": 1,  # calm
    "03": 2,  # happy
    "04": 3,  # sad
    "05": 4,  # angry
    "06": 5,  # fearful
    "07": 6,  # disgust
    "08": 7   # surprised
}


In [21]:
records = []

for root, _, files in os.walk(VIDEO_DIR):
    for f in files:
        if not f.lower().endswith(".mp4"):
            continue

        parts = f.replace(".mp4", "").split("-")
        if len(parts) != 7 or parts[0] != "01":
            continue

        img_path = os.path.join(IMAGE_DIR, f.replace(".mp4", ".jpg"))
        if not os.path.exists(img_path):
            continue

        records.append({
            "image": img_path,
            "emotion": emotion_map[parts[2]],
            "actor": int(parts[-1])
        })

df = pd.DataFrame(records)
len(df)


1440

In [22]:
train_df = df[df["actor"] <= 20].reset_index(drop=True)
val_df   = df[df["actor"] > 20].reset_index(drop=True)

len(train_df), len(val_df)


(1200, 240)

In [23]:
image_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5])
])


In [24]:
class ImageEmotionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image"]).convert("RGB")
        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row["emotion"], dtype=torch.long)
        return image, label


In [25]:
train_ds = ImageEmotionDataset(train_df, image_transform)
val_ds   = ImageEmotionDataset(val_df, image_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0    # 🔴 IMPORTANT for Windows
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0
)


In [26]:
img, label = train_ds[0]
img.shape, label


(torch.Size([3, 224, 224]), tensor(0))

In [27]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 8)
model = model.to(device)


In [28]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [29]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


In [30]:
@torch.no_grad()
def validate(model, loader):
    model.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total


In [31]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_acc = validate(model, val_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )


Epoch 1/10 | Loss: 1.5857 | Train Acc: 0.438 | Val Acc: 0.225
Epoch 2/10 | Loss: 0.5912 | Train Acc: 0.866 | Val Acc: 0.388
Epoch 3/10 | Loss: 0.2079 | Train Acc: 0.958 | Val Acc: 0.392
Epoch 4/10 | Loss: 0.0707 | Train Acc: 0.993 | Val Acc: 0.325
Epoch 5/10 | Loss: 0.0382 | Train Acc: 0.998 | Val Acc: 0.304
Epoch 6/10 | Loss: 0.0338 | Train Acc: 0.997 | Val Acc: 0.408
Epoch 7/10 | Loss: 0.0227 | Train Acc: 0.998 | Val Acc: 0.396
Epoch 8/10 | Loss: 0.0162 | Train Acc: 0.999 | Val Acc: 0.404
Epoch 9/10 | Loss: 0.0142 | Train Acc: 0.998 | Val Acc: 0.438
Epoch 10/10 | Loss: 0.0182 | Train Acc: 0.997 | Val Acc: 0.396
